# 12주차 실습 — E2E 의 현재와 미래: 어텐션 · 언어 조건부 · 미니 월드모델 · VLA 시연 · 자가 평가
**AI Application · 스마트모빌리티학과 · 2026-11-18**

| 셀 | 내용 |
|---|---|
| 0-1 | 이 머신의 속도 측정 → 예상 실행 시간, FAST |
| 1 | conv 스템 + 어텐션 1층 정책 — PilotNet 과 비교, 어텐션 맵(정책이 어느 열을 보나) |
| 2 | 언어 조건부 E2E — 8주차 CIL 의 명령을 문장으로 (FiLM), 새 문장·모르는 문장 일반화 |
| 3 | 미니 월드모델 — 상태 동역학 학습 → 상상으로 계획 (지평 H=1/5/20) |
| 4 | VLA 시연 로그 읽기 — OpenVLA-7B 지연·출력, 지연 예산 계산 |
| 5 | 자가 평가 — 내 정책의 L1~L5 → 가장 약한 레벨 |

**준비물**: `tinytrack.py` (0.3.1), Drive 의 `week11/my_policy_v3.onnx` 또는 `week07/my_policy_v2.pt` (없으면 즉석, `models/base_pilotnet.pt`). CPU 로 전체 6~8분. GPU 불필요.

In [ ]:
# ============ 0. 준비 (재접속 시 여기부터) ============
import importlib.util, subprocess, sys, os, time, json, copy
for pkg in ["onnx", "onnxruntime", "pandas"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import numpy as np, matplotlib.pyplot as plt, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
try:
    import tinytrack as tt
except ImportError:
    from google.colab import files
    print("tinytrack.py 를 업로드하세요"); files.upload(); import tinytrack as tt
assert tt.__version__ >= "0.3.1", "tinytrack 0.3.1 이상 필요"
try:
    from google.colab import drive
    drive.mount("/content/drive"); ROOT = Path("/content/drive/MyDrive/AIApp")
except Exception:
    ROOT = Path("./outputs")
SAVE_DIR = ROOT / "week12"; SAVE_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(0); np.random.seed(0)
MODELS = Path("models") if Path("models/base_pilotnet.pt").exists() else Path(".")

for cand in [ROOT / "week07" / "my_dataset_v2.npz", ROOT / "week05" / "my_dataset_v1.npz"]:
    if cand.exists():
        D = tt.load_npz(cand); print("데이터:", cand.name, len(D["steers"])); break
else:
    print("데이터가 없어 즉석 생성 (약 20초)")
    D = tt.rebalance(tt.merge(*[tt.collect(tt.ExpertPolicy(), tt.Track.preset(n), 500, seed=10 * ord(n) + k, noise_inject=ni)
                                for n in "ABC" for k, ni in enumerate([0.0, 0.4, 0.8])]))
D = {k: D[k] for k in ("images", "steers", "states", "run_id")}; tr, va = tt.split_by_run(D, 0.2, seed=0)
for cand in [ROOT / "week07" / "my_policy_v2.pt", ROOT / "week06" / "my_policy_v1.pt"]:
    if cand.exists():
        base = tt.load_model(cand); print("torch 정책:", cand.name); break
else:
    if not (MODELS / "base_pilotnet.pt").exists():
        from google.colab import files
        print("base_pilotnet.pt 업로드"); files.upload()
    base = tt.load_model(MODELS / "base_pilotnet.pt"); tt.train_bc(base, tr, va, epochs=3, lr=3e-4, augment=tt.make_augment(seed=0), verbose=False); print("torch 정책: 즉석 미세조정")
# 자가 평가용 ONNX: v3(11주차) → v2 내보내기
for cand in [ROOT / "week11" / "my_policy_v3.onnx", ROOT / "week10" / "ppo_best.onnx"]:
    if cand.exists():
        MY_ONNX = cand; break
else:
    MY_ONNX = tt.export_onnx(base, SAVE_DIR / "my_policy_current.onnx")
print("자가 평가 정책:", MY_ONNX)
PUBLIC = ["A", "B", "C"]
def batches(ds, idx, bs=64):
    for s in range(0, len(idx), bs):
        j = idx[s:s + bs]; yield j, torch.from_numpy(np.concatenate([tt.preprocess(im) for im in ds["images"][j]]))
def train_loop(model, ds, fwd, epochs=3, lr=3e-4, n=2500, seed=0, flip=True):       # fwd(model, x, j) → 예측. 명령 조건부 데이터는 flip=False (좌우 반전이 명령을 뒤집으므로)
    torch.manual_seed(seed); rng = np.random.RandomState(seed); opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr)
    idx_all = rng.choice(len(ds["steers"]), min(n, len(ds["steers"])), replace=False); ys = torch.from_numpy(ds["steers"]).unsqueeze(1); augf = tt.make_augment(flip=flip, seed=seed)
    for ep in range(epochs):
        model.train(); rng.shuffle(idx_all)
        for j, _ in batches(ds, idx_all):
            imgs, labs = zip(*[augf(ds["images"][k], float(ds["steers"][k])) for k in j])
            x = torch.from_numpy(np.concatenate([tt.preprocess(im) for im in imgs])); y = torch.tensor(labs, dtype=torch.float32).unsqueeze(1)
            loss = F.mse_loss(fwd(model, x, j), y); opt.zero_grad(); loss.backward(); opt.step()
    return model.eval()
print("tinytrack", tt.__version__, "| 프레임", len(D["steers"]), "| 저장:", SAVE_DIR)

### 0-1. 이 머신의 속도 측정 (5초) → 예상 실행 시간과 `FAST`

In [ ]:
import time as _t
_t0 = _t.time(); [tt.render(0.1, 0.05, 0.1) for _ in range(300)]; _tr = _t.time() - _t0
_m = tt.make_model().eval(); _x = torch.zeros(64, 3, tt.H, tt.W)
with torch.no_grad():
    _m(_x); _t0 = _t.time(); [_m(_x) for _ in range(10)]; _tf = _t.time() - _t0
REF_RENDER, REF_FWD, REF_MIN = 0.195, 0.318, 7
ratio = 0.5 * (_tr / REF_RENDER) + 0.5 * (_tf / REF_FWD)
FAST = os.environ.get("NB_FAST") == "1" or ratio > 1.8
N_TRAIN = 1500 if FAST else 2500
print(f"렌더 300장 {_tr:.2f}s · 순전파 10회 {_tf:.2f}s → 제작 PC 대비 {ratio:.1f}배 → 예상 전체 실행 약 {REF_MIN * ratio * (0.7 if FAST else 1.0):.0f}분 | FAST = {FAST}")

## 1. conv 스템 + 어텐션 1층 정책 — 정책이 어느 열을 보나
PilotNet conv 가 만든 (64 × 1 × 18) 특징을 **열 방향 토큰 18개**로 보고, 조향 토큰(CLS)이 어텐션으로 그것들을 읽습니다. 단일 헤드 어텐션을 직접 써서 **어텐션 맵**(조향 토큰이 어느 열을 얼마나 보나)을 그립니다.

In [ ]:
class AttnPolicy(nn.Module):
    def __init__(self, base, d=64):
        super().__init__(); self.stem = copy.deepcopy(base.features)
        self.cls = nn.Parameter(torch.zeros(1, 1, d)); self.pos = nn.Parameter(torch.zeros(1, 19, d) + 0.01 * torch.randn(1, 19, d))
        self.q, self.k, self.v = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.ff = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 128), nn.ReLU(), nn.Linear(128, d)); self.head = nn.Linear(d, 1); self.last_attn = None
    def forward(self, x):
        t = self.stem(x).flatten(2).transpose(1, 2)                              # (B,18,64)
        t = torch.cat([self.cls.expand(len(t), -1, -1), t], 1) + self.pos         # (B,19,64)
        A = torch.softmax(self.q(t) @ self.k(t).transpose(1, 2) / 8.0, dim=-1)   # (B,19,19)
        self.last_attn = A.detach()
        h = t + A @ self.v(t); h = h + self.ff(h)
        return self.head(h[:, 0])

t0 = time.time()
attn = train_loop(AttnPolicy(base), tr, lambda m, x, j: m(x), epochs=3, n=N_TRAIN)
pil = train_loop(copy.deepcopy(base), tr, lambda m, x, j: m(x), epochs=3, n=N_TRAIN)      # 같은 데이터·에폭으로 PilotNet 도 한 번 더
def mae(model, ds, n=400):
    idx = np.arange(min(n, len(ds["steers"])))
    with torch.no_grad(): pred = np.concatenate([model(x).numpy().ravel() for _, x in batches(ds, idx, 128)])
    return float(np.abs(pred - ds["steers"][idx]).mean())
rows = []
for name, m in [("PilotNet", pil), ("conv 스템 + 어텐션", attn)]:
    p = tt.export_onnx(m, SAVE_DIR / f"{name.replace(' ', '_')}.onnx"); op = tt.OnnxPolicy(p); op(D["images"][0]); t1 = time.time(); [op(im) for im in D["images"][:200]]
    ms = (time.time() - t1) / 200 * 1000                                                    # 지연은 평가 전에 확정
    rows.append(dict(model=name, params=sum(q.numel() for q in m.parameters()), offline_mae=round(mae(m, va), 4), L1=round(tt.evaluate(op, PUBLIC, seed=0)["score"], 1),
                     L2=round(tt.evaluate(op, PUBLIC, seed=0, **tt.level_eval_kwargs("L2"))["score"], 1), latency_ms=round(ms, 2)))
print(pd.DataFrame(rows).to_string(index=False), f"\n({time.time()-t0:.0f}s)")
# 어텐션 맵: 좌커브·직선·우커브 프레임에서 조향 토큰이 보는 열
fig, ax = plt.subplots(2, 3, figsize=(11, 3.2))
for i, (d0, p0, k0, ttl) in enumerate([(0.0, 0.0, 0.25, "left curve"), (0.0, 0.0, 0.0, "straight"), (0.0, 0.0, -0.25, "right curve")]):
    img = tt.render(d0, p0, k0)
    with torch.no_grad(): s = attn(torch.from_numpy(tt.preprocess(img))).item()
    ax[0, i].imshow(img); ax[0, i].set_title(f"{ttl}: steer {s:+.2f}", fontsize=9); ax[0, i].axis("off")
    ax[1, i].bar(range(18), attn.last_attn[0, 0, 1:].numpy(), color="#2563A8"); ax[1, i].set_ylim(0, 0.5); ax[1, i].set_xlabel("column token (left -> right)"); ax[1, i].set_title("attention of the steer token", fontsize=9)   # Colab 에는 한글 폰트가 없어 그림 글자는 영문
plt.tight_layout(); plt.savefig(SAVE_DIR / "attention_map.png", dpi=120); plt.show()
print("읽는 법: 커브에서 어텐션이 바깥 차선 쪽 열로 몰리면 '무엇을 보고 조향하는지' 가 보인다. 성능은 우리 규모(수천 장·3에폭)에서 PilotNet 이 앞서는 것이 정상 — conv 의 귀납 편향. 어텐션의 이득은 데이터가 많고 토큰 간 관계가 중요할 때")

## 2. 언어 조건부 E2E — 명령을 문장으로
8주차 CIL 데이터(왼쪽/중앙/오른쪽 차로 유지)를 다시 만들고, 명령 대신 **문장**을 조건으로 줍니다. 단어 임베딩 평균(bag-of-words) → FiLM 으로 특징을 조절. 학습 문장 / 단어가 겹치는 새 문장 / 모르는 단어 문장에서 **명령 추종 오차**를 재어 작은 언어 모델의 일반화 한계를 봅니다.

In [ ]:
class OffsetExpert:
    def __init__(self, target): self.target = target
    def __call__(self, obs, state): return tt.expert_steer(state[0] - self.target, state[1], state[2])
TARGET = {0: -0.25, 1: 0.0, 2: 0.25}
SENT = {0: ["keep left lane", "stay on the left side"], 1: ["stay center", "keep the middle lane"], 2: ["keep right lane", "stay on the right side"]}
TEST_NEW = {0: "drive on the left side", 1: "stay in the middle", 2: "drive on the right lane"}          # 단어가 겹치는 새 문장
TEST_UNK = {0: "go port side", 1: "hold your course", 2: "go starboard"}                                   # 모르는 단어만
t0 = time.time(); parts = []
for c, tgt in TARGET.items():
    ex = OffsetExpert(tgt); ds = tt.collect(ex, tt.Track.preset("B"), 500 if FAST else 600, seed=100 + c, noise_inject=0.4, expert=ex); ds["command"] = np.full(len(ds["steers"]), c); parts.append(ds)
C = {k: np.concatenate([p[k] for p in parts]) for k in ("images", "steers", "states", "command")}
words = sorted({w for ss in list(SENT.values()) + [list(TEST_NEW.values()), list(TEST_UNK.values())] for s in ss for w in s.lower().split()})
VOCAB = {w: i + 1 for i, w in enumerate(sorted({w for ss in SENT.values() for s in ss for w in s.lower().split()}))}   # 학습 문장의 단어만 어휘에
def tokens(s): return torch.tensor([VOCAB.get(w, 0) for w in s.lower().split()])                                       # 모르는 단어 → 0 (UNK)

class LangFiLMNet(nn.Module):
    def __init__(self, base, dim=16):
        super().__init__(); self.features = copy.deepcopy(base.features); self.emb = nn.Embedding(len(VOCAB) + 1, dim); self.emb.weight.data[0] = 0
        self.gamma = nn.Linear(dim, 64 * 18); self.beta = nn.Linear(dim, 64 * 18); self.head = nn.Sequential(nn.Linear(64 * 18, 50), nn.ReLU(), nn.Linear(50, 1))
    def lang(self, sentences):
        return torch.stack([self.emb(tokens(s)).mean(0) for s in sentences])
    def forward(self, x, sentences):
        f = self.features(x).flatten(1); l = self.lang(sentences)
        return self.head(f * (1 + self.gamma(l)) + self.beta(l))

rng = np.random.RandomState(0)
sent_of = [SENT[int(c)][rng.randint(2)] for c in C["command"]]                # 프레임마다 학습 문장 중 하나
net = train_loop(LangFiLMNet(base), C, lambda m, x, j: m(x, [sent_of[k] for k in j]), epochs=4, lr=1e-3, n=len(C["steers"]), flip=False)   # 좌우 반전 증강 금지!
class LangPolicy:
    def __init__(self, net, s): self.net, self.s = net, s
    def __call__(self, obs, state=None):
        with torch.no_grad(): return float(self.net(torch.from_numpy(tt.preprocess(obs)), [self.s]).item())
def mean_d(pol, steps=400, seed=0):
    env = tt.Env(tt.Track.preset("B"), seed=seed, max_steps=steps); obs = env.reset(seed=seed); ds_ = []
    for _ in range(steps):
        obs, _, term, trunc, info = env.step(pol(obs, env.state)); ds_.append(info["d"])
        if term or trunc: break
    return float(np.mean(ds_))
rows = []
for kind, sents in [("학습 문장", {c: SENT[c][0] for c in TARGET}), ("새 문장(단어 겹침)", TEST_NEW), ("모르는 단어", TEST_UNK)]:
    errs = {c: mean_d(LangPolicy(net, s)) - TARGET[c] for c, s in sents.items()}
    rows.append(dict(문장=kind, **{f"{['왼쪽','중앙','오른쪽'][c]} '{s}'": f"{TARGET[c] + errs[c]:+.2f}" for c, s in sents.items()}, 추종오차=round(float(np.mean([abs(e) for e in errs.values()])), 3)))
print(pd.DataFrame(rows).to_string(index=False), f"\n({time.time()-t0:.0f}s)")
print("읽는 법: 학습 문장 < 새 문장 < 모르는 단어 순으로 오차가 커진다. 어휘 밖 단어는 UNK → 평균으로 뭉개짐(명령 무시). VLA 가 사전학습 언어 모델을 쓰는 이유가 여기 있다 — 'port side' 가 왼쪽임을 이미 안다")

## 3. 미니 월드모델 — 상태 동역학을 배우고 상상으로 계획
무작위 행동으로 전이 `(d, ψ, κ, a) → (d', ψ')` 를 모아 MLP 동역학 모델을 학습하고, **행동 후보 21개를 H 스텝 상상**해 비용이 최소인 행동을 고릅니다(정책 없이 모델 + 탐색). 비용 = 매 스텝 |d| + 마지막 스텝 5·|d_H| (종단 비용). 지평 H 를 1/5/10/20 으로 바꿔 **근시안(H=1)** 과 **복합 오차(H=20)** 가 나타나는지 봅니다. (상태를 아는 계획기 — 실차라면 상태 추정기가 앞에 필요)

In [ ]:
t0 = time.time(); rng = np.random.RandomState(0); X, Y = [], []
for n in "ABC":
    env = tt.Env(tt.Track.preset(n), seed=1); env.reset(d0=0.0, psi0=0.0)
    for t in range(1500 if not FAST else 800):
        d, psi, kappa, _ = env.state; a = float(np.clip(tt.expert_steer(d, psi, kappa) + rng.normal(0, 0.5), -1, 1))   # 전문가 + 큰 노이즈 = 다양한 전이
        _, _, term, trunc, info = env.step(a); X.append([d, psi, kappa, a]); Y.append([info["d"], info["psi"]])
        if term or trunc: env.reset(d0=rng.uniform(-0.3, 0.3), psi0=rng.uniform(-0.3, 0.3))
X, Y = torch.tensor(X, dtype=torch.float32), torch.tensor(Y, dtype=torch.float32)
dyn = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2)); opt = torch.optim.Adam(dyn.parameters(), 3e-3)
for ep in range(300):
    perm = torch.randperm(len(X))
    for s in range(0, len(X), 256):
        j = perm[s:s + 256]; loss = F.mse_loss(dyn(X[j]), Y[j] - X[j, :2]); opt.zero_grad(); loss.backward(); opt.step()   # 잔차(변화량) 예측
with torch.no_grad(): res = (dyn(X) - (Y - X[:, :2])).abs().mean(0)
print(f"동역학 모델: 전이 {len(X)}개, 1스텝 오차 |Δd| {res[0]:.4f} |Δψ| {res[1]:.4f}  ({time.time()-t0:.0f}s)")

CANDS = torch.linspace(-1, 1, 21)
class PlanPolicy:                                                  # 상상 H 스텝 → 비용 최소 후보 (곡률은 현재값 유지 가정)
    def __init__(self, dyn, H): self.dyn, self.H = dyn, H
    def __call__(self, obs, state):
        with torch.no_grad():
            d = torch.full((21,), float(state[0])); psi = torch.full((21,), float(state[1])); kappa = torch.full((21,), float(state[2])); cost = torch.zeros(21)
            for h in range(self.H):
                delta = self.dyn(torch.stack([d, psi, kappa, CANDS], 1)); d, psi = d + delta[:, 0], psi + delta[:, 1]
                cost += d.abs() + 0.02 * CANDS.abs() + (5.0 * d.abs() if h == self.H - 1 else 0.0)     # 종단 비용: 지평 끝에서 중앙에 있어야
            return float(CANDS[cost.argmin()])
rows = []
for H in (1, 5, 10, 20):
    t1 = time.time(); r = tt.evaluate(PlanPolicy(dyn, H), PUBLIC, seed=0); rows.append(dict(H=H, score=round(r["score"], 1), completion=r["completion"], mean_abs_d=round(r["mean_abs_d"], 3), sec=round(time.time() - t1)))
rows.append(dict(H="expert", score=round(tt.evaluate(tt.ExpertPolicy(), PUBLIC, seed=0)["score"], 1), completion=1.0, mean_abs_d=None, sec=0))
print(pd.DataFrame(rows).to_string(index=False))
print("읽는 법: H=1 은 근시안(다음 스텝의 d 는 행동과 무관해 헤딩만 맞춤) → 실패. H=5~10 이 최선(전문가를 넘기도), H=20 은 모델 오차가 쌓여 나빠진다 — 복합 오차")

## 4. VLA 시연 로그 읽기 — OpenVLA-7B (교수 GPU, 2026-09)
GPU 없이 결과 로그만 읽습니다. 로드·VRAM·지연·출력 벡터를 보고 **지연 예산**을 계산합니다.

In [ ]:
VLA_LOG = '''model=openvla/openvla-7b device=cuda | GPU NVIDIA RTX 6000 Ada Generation 47 GB
loaded in 17s | VRAM 14.1 GB
prompt: "Drive along the lane and keep the car centered."
predict_action #0: 1884 ms → action(7-DoF, bridge_orig 정규화 해제) = [-0.005 -0.01  -0.016  0.     -0.007  0.003  0.   ]
predict_action #1:  233 ms → action = [-0.005 -0.01  -0.016  0.     -0.007  0.003  0.   ]
predict_action #2:  234 ms → action = [-0.005 -0.01  -0.016  0.     -0.007  0.003  0.   ]
prompt: "Turn left at the next corner."
predict_action #1:  223 ms → action = [-0.025 -0.009 -0.026 -0.081  0.078  0.04   0.   ]'''
print(VLA_LOG)
ms_vla, ms_pilot = 230, 1.0
for name, ms in [("OpenVLA-7B (GPU 서버)", ms_vla), ("PilotNet ONNX (노트북 CPU)", ms_pilot)]:
    print(f"{name:28s}: {ms:6.1f} ms/프레임 → {1000/ms:6.1f} fps | 0.5 m/s 에서 결정 간격 {0.5*ms/1000*100:5.1f} cm | 33 ms 예산의 {ms/33:.2f}배")
print("읽는 법: 7-DoF 팔 사전분포라 차 이미지엔 값이 거의 0 — 그래도 언어('turn left')에 따라 회전 성분이 바뀐다 = 구조는 동작. 230 ms 는 33 ms 루프의 7배 → VLA 는 상위 계획기, 33 ms 제어는 작은 E2E 가 (계층 구조)")

## 5. 자가 평가 — 내 정책의 L1~L5
10·11주차에 쓴 레벨 그대로입니다. 여기서는 **공개 트랙 + 시드 0** 만 봅니다 — 미공개 트랙·다른 스타일 시드에서는 점수가 달라질 수 있습니다(특히 색 랜덤화가 있는 L3~L5).

In [ ]:
pol = tt.OnnxPolicy(MY_ONNX); t0 = time.time(); lv = {}
for L in ("L1", "L2", "L3", "L4", "L5"):
    r = tt.evaluate(pol, PUBLIC, seed=0, **tt.level_eval_kwargs(L, style_seed=7)); lv[L] = round(r["score"], 1)
print(lv, f"({time.time()-t0:.0f}s)")
weak = min(("L2", "L3", "L4"), key=lambda k: lv[k])
print(f"가장 약한 레벨: {weak} ({tt.CHALLENGE_LEVELS[weak]['name']}) — 오늘까지 배운 처방 중 무엇이 이 레벨에 맞을까? (외란→DAgger, 저조도→랜덤화, 고속→RL·속도 랜덤화)")
json.dump(dict(levels=lv, policy=str(MY_ONNX)), open(SAVE_DIR / "self_eval.json", "w"), indent=1)

## 정리 · 제출
- 어텐션은 '무엇을 볼지'를 데이터가 정하게 한다 — 우리 규모(수천 장)에선 PilotNet 이 앞서고, 어텐션 맵이 해석을 준다.
- 언어 조건은 어휘 안에서만 통한다 — 사전학습 언어 모델(VLA)의 값어치. VLA 의 230 ms 는 계층 구조를 강제한다.
- 월드모델 계획은 정책 없이도 달리지만 지평이 길면 오차가 쌓인다.
- 내 정책의 가장 약한 레벨을 기록해 두세요 (`week12/self_eval.json`).